# D2 - Instacart Market Basket Analysis: Luật kết hợp + Gom cụm

## 1. Nguồn - giấy phép - quy mô

| Mục | Thông tin |
|---|---|
| Dataset | Instacart Market Basket Analysis |
| Link | https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis |
| Đơn vị công bố | Instacart (qua Kaggle) |
| Giấy phép | CC BY-NC-SA 4.0 |
| Ngày tải | 2024 (dữ liệu cập nhật năm 2017) |
| Quy mô công bố | khoảng 3.4 triệu dòng trong `order_products`; nhiều bảng liên kết, hơn 34 thuộc tính |
| Kỹ thuật yêu cầu | **Luật kết hợp + Gom cụm** |

**Tri thức lĩnh vực:** mỗi đơn hàng gồm nhiều sản phẩm. Luật kết hợp giúp tìm các sản phẩm thường được mua cùng nhau; các đặc trưng tổng hợp theo khách hàng như số đơn, số sản phẩm và chu kỳ mua lại phù hợp để phân khúc khách hàng bằng gom cụm.

## 2. Từ điển dữ liệu

| Bảng/thuộc tính | Ý nghĩa | Kiểu | Thang đo |
|---|---|---|---|
| `orders.order_id` | Mã đơn hàng | int | định danh |
| `orders.user_id` | Mã khách hàng | int | định danh |
| `orders.order_number` | Thứ tự đơn của khách hàng | int | tỷ lệ |
| `orders.days_since_prior_order` | Số ngày từ đơn trước | float | tỷ lệ |
| `order_products.product_id` | Mã sản phẩm | int | định danh |
| `order_products.add_to_cart_order` | Thứ tự thêm vào giỏ | int | thứ hạng |
| `order_products.reordered` | Sản phẩm có được mua lại | int | nhị phân |
| `products.product_name` | Tên sản phẩm | string | danh nghĩa |
| `products.aisle_id`, `products.department_id` | Ngành hàng và khu vực | int | danh nghĩa |

Ma trận kỹ thuật: **D2 = Luật kết hợp + Gom cụm**.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ROOT = Path(r'C:\data-mining-project\vs-code-interface-review\bai1-du-lieu-tien-xu-ly\D2')
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'outputs'
FIG = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(exist_ok=True)
SAMPLE_ROWS = 200000

def find_file(name):
    matches = list(RAW.rglob(name))
    if not matches:
        raise FileNotFoundError(f'Không tìm thấy {name} trong {RAW}')
    return matches[0]

orders = pd.read_csv(find_file('orders.csv'))
order_products = pd.read_csv(find_file('order_products__prior.csv'), nrows=SAMPLE_ROWS)
products = pd.read_csv(find_file('products.csv'))
print('orders:', orders.shape, '| order_products: đọc theo từng chunk | products:', products.shape)

## 3. Khám phá dữ liệu và giá trị thiếu

Kiểm tra kiểu dữ liệu, khóa liên kết và tỷ lệ thiếu trước khi biến đổi. Các bản ghi thiếu `product_name` hoặc khóa liên kết không được dùng cho luật kết hợp.

In [ ]:
missing = pd.concat({
    'orders': orders.isna().sum(),
    'order_products': order_products.isna().sum(),
    'products': products.isna().sum()
}, axis=1).fillna(0)
missing.to_csv(OUT / 'missing_report.csv', encoding='utf-8-sig')
missing['total_missing'] = missing.sum(axis=1)
display(missing[missing['total_missing'] > 0].sort_values('total_missing', ascending=False).head(20))
assert orders['order_id'].is_unique
assert products['product_id'].is_unique


## 4. Tiền xử lý và biến đổi thuộc tính

Ghép đơn hàng với sản phẩm, loại bản ghi không hợp lệ, sau đó tạo đặc trưng hành vi theo khách hàng. Vì dữ liệu lớn, luật kết hợp được chạy trên các sản phẩm phổ biến và một mẫu đơn hàng có kiểm soát.

In [ ]:
order_user = orders[['order_id', 'user_id']].drop_duplicates()
stats = []
for chunk in pd.read_csv(find_file('order_products__prior.csv'), usecols=['order_id', 'product_id', 'add_to_cart_order', 'reordered'], nrows=SAMPLE_ROWS, chunksize=100000):
    chunk = chunk.merge(order_user, on='order_id', how='inner')
    stats.append(chunk.groupby('user_id').agg(orders=('order_id', 'nunique'), products=('product_id', 'count'), unique_products=('product_id', 'nunique'), reorder_sum=('reordered', 'sum'), cart_sum=('add_to_cart_order', 'sum'), cart_count=('add_to_cart_order', 'count')).reset_index())
summary = pd.concat(stats).groupby('user_id').agg(orders=('orders', 'sum'), products=('products', 'sum'), unique_products=('unique_products', 'sum'), reorder_sum=('reorder_sum', 'sum'), cart_sum=('cart_sum', 'sum'), cart_count=('cart_count', 'sum')).reset_index()
customer_features = summary.assign(reorder_rate=summary['reorder_sum'] / summary['products'], mean_cart_position=summary['cart_sum'] / summary['cart_count']).drop(columns=['reorder_sum', 'cart_sum', 'cart_count'])
detail_sample = pd.read_csv(find_file('order_products__prior.csv'), nrows=SAMPLE_ROWS)
detail = (detail_sample.merge(orders[['order_id', 'user_id']], on='order_id', how='inner').merge(products[['product_id', 'product_name']], on='product_id', how='left').dropna(subset=['product_name']))
detail['product_name'] = detail['product_name'].str.strip()
customer_features.to_csv(PROCESSED / 'customer_features.csv', index=False, encoding='utf-8-sig')
numeric = ['orders', 'products', 'unique_products', 'reorder_rate', 'mean_cart_position']
z = np.abs((customer_features[numeric] - customer_features[numeric].mean()) / customer_features[numeric].std())
outliers = pd.DataFrame({'attribute': numeric, 'count_z_gt_3': (z > 3).sum().values})
outliers.to_csv(OUT / 'outlier_report.csv', index=False, encoding='utf-8-sig')
display(customer_features.head())


## 5. Luật kết hợp

Mỗi `order_id` là một giao dịch và mỗi sản phẩm là một item. Dùng sản phẩm phổ biến để giới hạn ma trận one-hot; ngưỡng có thể điều chỉnh theo máy chạy.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

top_products = detail['product_id'].value_counts().head(200).index
transactions = (detail[detail['product_id'].isin(top_products)]
                .groupby('order_id')['product_name'].apply(lambda x: list(set(x))).tolist())
transactions = transactions[:SAMPLE_ROWS]
te = TransactionEncoder()
basket = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
frequent = apriori(basket, min_support=0.01, use_colnames=True, max_len=3)
rules = association_rules(frequent, metric='confidence', min_threshold=0.2)
rules = rules.sort_values(['lift', 'confidence'], ascending=False)
rules.to_csv(OUT / 'association_rules.csv', index=False, encoding='utf-8-sig')
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20))


## 6. Gom cụm khách hàng và đánh giá

Chuẩn hóa đặc trưng trước K-Means. Chọn số cụm có silhouette cao trong khoảng thử nghiệm.

In [ ]:
X = StandardScaler().fit_transform(customer_features[numeric].fillna(0))
scores = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    sample_idx = np.random.RandomState(42).choice(len(X), size=min(5000, len(X)), replace=False)
    scores.append({'k': k, 'inertia': model.inertia_, 'silhouette': silhouette_score(X[sample_idx], labels[sample_idx])})
scores = pd.DataFrame(scores)
scores.to_csv(OUT / 'clustering_scores.csv', index=False)
best_k = int(scores.loc[scores['silhouette'].idxmax(), 'k'])
customer_features['cluster'] = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X)
customer_features.to_csv(OUT / 'customer_clusters.csv', index=False, encoding='utf-8-sig')
display(scores)
display(customer_features.groupby('cluster')[numeric].mean().round(2))
